In [1]:
# Imports
import pandas as pd
import numpy as np


In [2]:
# Load train and validation sets, parsing Time as datetime
train_df = pd.read_csv('../Data/train.csv')
val_df = pd.read_csv('../Data/val.csv')

train_df['Time'] = pd.to_datetime(train_df['Time'], format='mixed', errors='coerce')
val_df['Time'] = pd.to_datetime(val_df['Time'], format='mixed', errors='coerce')


In [3]:
# Sort by Time to guarantee chronological order before creating lag features
train_df = train_df.sort_values('Time').reset_index(drop=True)
val_df = val_df.sort_values('Time').reset_index(drop=True)


In [4]:
# Create lag and rolling features from Power[kW] on each DataFrame separately
dfs = [train_df, val_df]
for df in dfs:
    df['power_lag1'] = df['Power[kW]'].shift(1)
    df['power_lag2'] = df['Power[kW]'].shift(2)
    df['power_lag4'] = df['Power[kW]'].shift(4)
    df['power_roll4'] = df['Power[kW]'].rolling(window=4).mean()


In [5]:
# Create forecasting targets for the 15-minute and 1-hour horizons
for df in dfs:
    df['target_15min'] = df['Power[kW]'].shift(-1)
    df['target_1hour'] = df['Power[kW]'].shift(-4)


In [6]:
# Drop rows with NaN values created by the lag/shift operations
train_rows_before = len(train_df)
val_rows_before = len(val_df)

train_df = train_df.dropna().reset_index(drop=True)
val_df = val_df.dropna().reset_index(drop=True)

train_rows_after = len(train_df)
val_rows_after = len(val_df)

print('Train rows before:', train_rows_before, 'after:', train_rows_after)
print('Val rows before:', val_rows_before, 'after:', val_rows_after)


Train rows before: 71647 after: 71639
Val rows before: 17735 after: 17727


In [7]:
# Verify alignment: target_15min at row N equals Power[kW] at row N+1, target_1hour at row N equals Power[kW] at row N+4
check_cols = ['Time', 'Power[kW]', 'power_lag1', 'target_15min', 'target_1hour']
print(train_df[check_cols].head(6))


                 Time  Power[kW]  power_lag1  target_15min  target_1hour
0 2017-01-01 08:15:00      0.132       0.020         0.176         0.332
1 2017-01-01 08:30:00      0.176       0.132         0.244         0.276
2 2017-01-01 08:45:00      0.244       0.176         0.260         0.392
3 2017-01-01 09:00:00      0.260       0.244         0.332         0.552
4 2017-01-01 09:15:00      0.332       0.260         0.276         0.644
5 2017-01-01 09:30:00      0.276       0.332         0.392         0.476


In [8]:
# Regression metrics helper
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_report(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{label}  RMSE={rmse:,.3f}  MAE={mae:,.3f}  R2={r2:.3f}')
    return rmse, mae, r2


In [9]:
# Persistence baseline: predict the future value as the current Power[kW]
persistence_pred_15min = val_df['Power[kW]']
persistence_pred_1hour = val_df['Power[kW]']

rmse_15min, mae_15min, r2_15min = regression_report(val_df['target_15min'], persistence_pred_15min, label='15-min persistence baseline')
rmse_1hour, mae_1hour, r2_1hour = regression_report(val_df['target_1hour'], persistence_pred_1hour, label='1-hour persistence baseline')


15-min persistence baseline  RMSE=1.723  MAE=0.925  R2=0.863
1-hour persistence baseline  RMSE=2.895  MAE=1.890  R2=0.614


In [10]:
# Save the prepared forecasting datasets
train_df.to_csv('../Data/train_forecast.csv', index=False)
val_df.to_csv('../Data/val_forecast.csv', index=False)


In [11]:
# Print final shape and column list for both saved DataFrames
print('train_forecast shape:', train_df.shape)
print('train_forecast columns:', list(train_df.columns))
print('val_forecast shape:', val_df.shape)
print('val_forecast columns:', list(val_df.columns))


train_forecast shape: (71639, 25)
train_forecast columns: ['Time', 'GHI', 'temp', 'pressure', 'humidity', 'wind_speed', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_type', 'sunlightTime', 'dayLength', 'SunlightTime/daylength', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'isSun', 'Power[kW]', 'power_lag1', 'power_lag2', 'power_lag4', 'power_roll4', 'target_15min', 'target_1hour']
val_forecast shape: (17727, 25)
val_forecast columns: ['Time', 'GHI', 'temp', 'pressure', 'humidity', 'wind_speed', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_type', 'sunlightTime', 'dayLength', 'SunlightTime/daylength', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'isSun', 'Power[kW]', 'power_lag1', 'power_lag2', 'power_lag4', 'power_roll4', 'target_15min', 'target_1hour']
